# Kaggriculture Meta Census — Lineage Occupancy

Who is farming what on the top of the ladder, and how fast does the meta rotate?

This notebook tracks **agent lineages** in the official
[Kaggriculture episodes datasets](https://www.kaggle.com/datasets/kaggle/kaggriculture-episodes-index)
(a daily excerpt of top-band games) and charts their **occupancy over time**.

## Method

- A seat's **lineage** is the SHA-256 (first 10 hex chars) of its first 24 steps
  of actions — the "opening signature". Deterministic agents produce a stable
  signature across towns and seeds, so identical-opening agents cluster
  together even when their mid-game behavior adapts.
- **Occupancy** = share of seats per day carrying each signature.
- A lineage is **labeled with a public notebook's name only when that
  notebook's extracted agent reproduces the signature in a local mirror match**
  (a verifiable claim anyone can replicate). Everything else stays a bare hash
  — no guessing about private agents' origins.
- Earlier days are carried as pre-computed counts (same pipeline, run daily);
  the attached day is mined live from the raw episode JSONs below.

The result is a picture of the meta as an ecosystem: public-notebook waves
sweeping the ladder in days, counter-strategies rising, and the occasional
unlabeled newcomer climbing out of nowhere.


In [ ]:
# Kaggriculture meta census (public edition).
#
# Mines the attached daily episodes dataset into a per-seat lineage census,
# merges it with the carried history, and renders the occupancy chart inline.
# Outputs:
#   census-<date>.json  — one row per episode: town, rewards, teams, lineages
#   census_trend.html   — the same interactive chart as a standalone file

import glob
import hashlib
import json
import os
import re

# Lineages labeled by their source notebook — ONLY where the notebook's
# extracted agent reproduces the opening signature in a local mirror match.
# Everything else is shown as its bare signature hash.
NOTEBOOK_LABELS = {
    "bb7fbea029": "v54 Strict Flow (Kaito Fukami)",
    "ee954925d4": "Shape the Shop Work the Pasture (Indar Karhana)",
    "cb8e6be117": "Multi-Route Farming Agent (flexonafft)",
}

# Daily seat counts per lineage from previous runs of this same pipeline.
# --- nbmine:history:start ---
HISTORY = {"2026-08-23": {"0336bbe1df": 134, "040ffdf2ed": 27, "2b01a546d4": 90, "2f54686340": 40, "393c72dd52": 35, "47a038ebea": 4, "5146031dc6": 24, "62c67bbd0d": 20, "758d7736d0": 60, "7e8222dcd3": 4, "8786c0d4da": 38, "95f4f78d90": 59, "af2d00c1a8": 7, "baa3f76dbf": 311, "c831787799": 20, "cb8e6be117": 478, "e66b86ee81": 12, "e838598664": 9, "ffd5f2f569": 12}, "2026-08-24": {"0336bbe1df": 48, "09970d9f67": 25, "2b01a546d4": 50, "2f54686340": 36, "5146031dc6": 20, "62c67bbd0d": 1, "758d7736d0": 7, "8786c0d4da": 5, "9410a69601": 1, "95f4f78d90": 16, "baa3f76dbf": 1024, "c831787799": 24, "cb8e6be117": 121, "e66b86ee81": 3, "ffd5f2f569": 3}, "2026-08-25": {"0336bbe1df": 6, "09970d9f67": 34, "2b01a546d4": 48, "2f54686340": 29, "50bead8d1b": 10, "5146031dc6": 37, "62c67bbd0d": 1, "95f4f78d90": 4, "99f2f6f4c6": 10, "9c096cd3cf": 3, "baa3f76dbf": 1102, "c2ebeb3bf2": 18, "c831787799": 22, "cb8e6be117": 7, "f30d2561bb": 43, "ffd5f2f569": 2}, "2026-08-26": {"09970d9f67": 24, "0a33371cdf": 1, "2b01a546d4": 45, "2b3268ddc5": 10, "2f54686340": 29, "50bead8d1b": 94, "5146031dc6": 55, "9410a69601": 1, "97d88078fd": 3, "99f2f6f4c6": 32, "9c096cd3cf": 11, "baa3f76dbf": 827, "bb7fbea029": 12, "be0a8cd059": 81, "c2ebeb3bf2": 52, "c831787799": 17, "c9713d0283": 2, "f30d2561bb": 68, "ffd5f2f569": 10}, "2026-08-27": {"09970d9f67": 33, "0a33371cdf": 8, "10ac811f26": 1, "12a91d93c6": 2, "2b01a546d4": 36, "2b3268ddc5": 7, "2f54686340": 35, "50bead8d1b": 492, "5146031dc6": 38, "77f1277f9a": 2, "889de90d84": 44, "8a94a2d4a7": 55, "97d88078fd": 3, "99f2f6f4c6": 34, "9c096cd3cf": 48, "baa3f76dbf": 333, "bb7fbea029": 69, "be0a8cd059": 30, "c2ebeb3bf2": 29, "c831787799": 9, "d3ef5fb203": 15, "f30d2561bb": 55, "ffd5f2f569": 14}, "2026-08-28": {"09970d9f67": 34, "0a33371cdf": 2, "1099c6ffc7": 12, "10ac811f26": 1, "12a91d93c6": 5, "2b01a546d4": 1, "2d9e4b260e": 5, "2f54686340": 36, "42ed29929f": 32, "50bead8d1b": 215, "5146031dc6": 41, "77f1277f9a": 15, "889de90d84": 14, "8a94a2d4a7": 78, "97d88078fd": 23, "99f2f6f4c6": 42, "9e6e9a13f0": 55, "aca7211231": 10, "af2d00c1a8": 1, "b12654cf75": 26, "baa3f76dbf": 94, "bb7fbea029": 477, "be0a8cd059": 3, "c2ebeb3bf2": 8, "c831787799": 26, "c9713d0283": 9, "d2b815f0c3": 57, "d3ef5fb203": 4, "f30d2561bb": 18, "ffd5f2f569": 8}, "2026-08-29": {"09970d9f67": 39, "2d9e4b260e": 1, "2f54686340": 1, "42ed29929f": 7, "50bead8d1b": 26, "5146031dc6": 35, "77f1277f9a": 21, "891495828b": 54, "8a94a2d4a7": 9, "99f2f6f4c6": 22, "9e6e9a13f0": 7, "b12654cf75": 48, "baa3f76dbf": 6, "bb7fbea029": 774, "be0a8cd059": 1, "c831787799": 31, "c9713d0283": 9, "d2b815f0c3": 60, "ee954925d4": 76, "f30d2561bb": 104, "ffd5f2f569": 3}, "2026-08-30": {"02f2165577": 10, "09970d9f67": 12, "42ed29929f": 15, "5146031dc6": 36, "548839bf65": 1, "5a5bb07cde": 29, "77f1277f9a": 9, "891495828b": 212, "a8290df987": 2, "b12654cf75": 33, "b93635aa37": 12, "bb7fbea029": 509, "c831787799": 25, "c9713d0283": 10, "d2b815f0c3": 43, "e7f89965a4": 3, "e9c9700032": 96, "ee954925d4": 192, "f30d2561bb": 79}, "2026-08-31": {"02f2165577": 19, "1f6a8a5ba5": 1, "42ed29929f": 21, "5146031dc6": 9, "548839bf65": 7, "5a5bb07cde": 62, "7529e16a57": 41, "891495828b": 220, "9e6e9a13f0": 30, "a8290df987": 55, "b12654cf75": 191, "b93635aa37": 13, "bb7fbea029": 95, "c831787799": 22, "e707c6c560": 2, "e9c9700032": 384, "ee954925d4": 68, "f30d2561bb": 90}, "2026-09-01": {"15d5638b8b": 41, "198d4ef8c5": 1, "1fa5c13552": 2, "407677684b": 3, "4a9a71b75c": 3, "5a5bb07cde": 83, "5ccb3e7419": 1, "7529e16a57": 44, "7a32851180": 114, "891495828b": 139, "8cf2dce726": 1, "96d6a4d4f1": 4, "9e6e9a13f0": 94, "a8290df987": 44, "b12654cf75": 245, "b64a38a635": 30, "b7d0048499": 24, "bb7fbea029": 202, "c831787799": 18, "cd7c22f878": 3, "e023a4729d": 21, "e9c9700032": 193, "f30d2561bb": 11, "fbb2028994": 7}, "2026-09-02": {"0f84ed7ca9": 7, "15d5638b8b": 126, "1ed64ee8b6": 3, "407677684b": 3, "4a66ccad44": 102, "4a9a71b75c": 3, "5167ef9b2e": 5, "572f448235": 2, "5a5bb07cde": 15, "7529e16a57": 18, "7584ae92d2": 37, "7a32851180": 15, "82a3c72c31": 411, "8cf2dce726": 106, "921500c57a": 62, "9e6e9a13f0": 52, "a12bc717bd": 5, "a2113ce935": 44, "a3d541669f": 19, "ad83b72700": 14, "af2d001140": 15, "b7d0048499": 156, "bde878b74a": 34, "be07a50e8c": 9, "c6fdb6c947": 38, "cbfd9c5bd8": 12, "eb04e30493": 5, "f99f4470a7": 1, "fbb2028994": 1}, "2026-09-03": {"02b953cf3f": 5, "0d2053e2f3": 23, "15ebeb5076": 15, "206e0ca52b": 16, "2a9aa2f7b6": 64, "313fdee774": 22, "4853c19a6d": 1, "4a66ccad44": 8, "567b3c1670": 89, "572f448235": 97, "6af7240248": 16, "7584ae92d2": 24, "801d626ba6": 76, "82a3c72c31": 25, "920558c65e": 24, "921500c57a": 110, "9e6e9a13f0": 95, "a2113ce935": 25, "ad83b72700": 2, "b17f601b80": 1, "b1bed42093": 225, "b7d0048499": 89, "b9806b1857": 65, "bb7fbea029": 1, "c6fdb6c947": 111, "cbfd9c5bd8": 14, "d7ab0eb7a8": 8, "e3ce37726f": 1, "e8b43f4151": 17, "e9916a8a7b": 8, "f336a49e19": 43, "f866a404d2": 18}, "2026-09-04": {"02b953cf3f": 17, "0d2053e2f3": 99, "15d5638b8b": 47, "15ebeb5076": 61, "206e0ca52b": 238, "2a9aa2f7b6": 1, "313fdee774": 71, "4853c19a6d": 89, "567b3c1670": 3, "572f448235": 16, "630f595e62": 11, "6737318541": 1, "67dd108dc6": 8, "6af7240248": 40, "6e7e6b506f": 13, "751741f2aa": 3, "7dafab486a": 1, "801d626ba6": 125, "8663bc481a": 20, "8a6fb90873": 16, "921500c57a": 19, "9e4772a04e": 1, "9e6e9a13f0": 146, "a2113ce935": 35, "b1bed42093": 36, "b7d0048499": 74, "b9806b1857": 3, "bb7fbea029": 2, "c0ddc7a55d": 1, "c199b8d99c": 1, "ca72492e70": 8, "cc587fe77e": 3, "d470b908f3": 2, "d7ab0eb7a8": 30, "d95b21c65e": 86, "e9916a8a7b": 9}, "2026-09-05": {"02b953cf3f": 11, "0cc9495557": 42, "0ee407b42a": 1, "1305c9267e": 1, "15d5638b8b": 137, "15ebeb5076": 24, "2029546c36": 9, "206e0ca52b": 236, "2ef23e0c6e": 69, "313fdee774": 8, "4853c19a6d": 65, "509124e958": 1, "56d01571c4": 2, "5a3af96ed2": 7, "630f595e62": 12, "6b77ea5fff": 19, "6e7e6b506f": 38, "74fee8f305": 29, "751741f2aa": 17, "773fc1dc20": 1, "801d626ba6": 56, "864d69b2c9": 6, "891495828b": 91, "89f66cc8f0": 4, "8a6fb90873": 2, "921500c57a": 14, "92d4bd36a9": 1, "9949c651af": 1, "9e4772a04e": 26, "a36d6bd8ff": 45, "a9f3eb7993": 3, "b1bed42093": 4, "b9806b1857": 5, "bb7fbea029": 32, "c199b8d99c": 1, "ca72492e70": 34, "cf45f8d309": 1, "d24b793042": 102, "d7ab0eb7a8": 3, "d95b21c65e": 143, "e4d1dcd188": 24, "ef0a639803": 1}, "2026-09-06": {"02b953cf3f": 13, "0987ccc1cf": 1, "0cc9495557": 173, "0d7a70228a": 29, "0e5dce2d95": 63, "0ee407b42a": 158, "0f76e5a717": 1, "15d5638b8b": 12, "1878c75f8b": 8, "1daf3a51f6": 9, "206e0ca52b": 26, "2519e4b820": 4, "27e2775248": 2, "2ef23e0c6e": 31, "333016335f": 66, "368452c2f0": 15, "390c1f4ce0": 2, "3f7aa60113": 20, "448d220e4a": 1, "4853c19a6d": 47, "53d692273b": 5, "56d01571c4": 43, "5a3af96ed2": 53, "5bdb79ed02": 2, "6ba133b6a7": 1, "6d0ab02ebf": 34, "6e7e6b506f": 58, "773fc1dc20": 2, "8041ec8e4a": 16, "8338da50ac": 23, "864d69b2c9": 2, "86c798fec9": 1, "89f66cc8f0": 5, "8a6fb90873": 5, "8b230b25e9": 60, "9482924619": 1, "96eb9e38d4": 1, "98895ba5f7": 2, "9ae83b19d2": 8, "9e4772a04e": 1, "a36d6bd8ff": 43, "a47498b650": 1, "a9f3eb7993": 1, "b9806b1857": 19, "bafed64435": 1, "cf45f8d309": 1, "d24b793042": 75, "d6aae107c3": 1, "d95b21c65e": 15, "da9d90b9e5": 13, "dec95004aa": 5, "defd41a32d": 1, "e4d1dcd188": 134, "f0911bc98e": 1, "f0995b3733": 2, "f18b646c68": 2, "f1e8f23f2d": 4, "f71b6ee2bf": 1}}
# --- nbmine:history:end ---

OUT = "/kaggle/working"


def lineage_sig(steps, seat):
    """Opening signature: hash of the seat's first 24 steps of actions."""
    stream = []
    for t in range(24):
        entry = steps[t + 1][seat] if len(steps) > t + 1 else None
        stream.append((entry or {}).get("action") or {})
    return hashlib.sha256(json.dumps(stream, sort_keys=True).encode()).hexdigest()[:10]


def mine_date(date, files):
    census = []
    for k, path in enumerate(sorted(files)):
        eid = int(os.path.basename(path).split(".")[0])
        try:
            rep = json.load(open(path))
        except Exception as e:
            print(f"[{k}] {eid} parse fail {e}", flush=True)
            continue
        steps = rep.get("steps") or []
        if len(steps) < 700:
            continue
        shops = steps[-1][0]["observation"]["town"]["unlocked_shops"]
        census.append({
            "episode": eid, "date": date,
            "town": shops[0] if shops else None,
            "rewards": rep.get("rewards") or [None, None],
            "teams": (rep.get("info") or {}).get("TeamNames"),
            "lineages": [lineage_sig(steps, 0), lineage_sig(steps, 1)],
        })
        if k % 100 == 0:
            print(f"[{k}/{len(files)}] ep{eid}", flush=True)
    with open(f"{OUT}/census-{date}.json", "w") as fh:
        json.dump(census, fh)
    print(f"{date}: {len(census)} episodes", flush=True)
    return census


def render_trend(history):
    """Stacked-area occupancy chart, rendered inline + saved as HTML."""
    import plotly.graph_objects as go
    import plotly.io as pio

    pio.renderers.default = "notebook"  # embed plotly.js in the cell output

    dates = sorted(history)
    if not dates:
        return
    # Pick the majors: overall presence, weighted toward the most recent day.
    total = {}
    for d in dates:
        n = sum(history[d].values()) or 1
        for lin, k in history[d].items():
            total[lin] = total.get(lin, 0) + k / n
    recent = history[dates[-1]]
    rn = sum(recent.values()) or 1
    score = {lin: total[lin] + 3 * recent.get(lin, 0) / rn for lin in total}
    majors = sorted(score, key=score.get, reverse=True)[:9]

    fig = go.Figure()
    ns = {d: sum(history[d].values()) or 1 for d in dates}
    acc = {d: 0.0 for d in dates}
    for m in majors:
        ys = [history[d].get(m, 0) / ns[d] * 100 for d in dates]
        for d, y in zip(dates, ys):
            acc[d] += y
        fig.add_trace(go.Scatter(
            x=dates, y=ys, name=NOTEBOOK_LABELS.get(m, m), stackgroup="one",
            mode="lines", hovertemplate="%{y:.1f}%"))
    fig.add_trace(go.Scatter(
        x=dates, y=[max(0.0, 100.0 - acc[d]) for d in dates], name="others",
        stackgroup="one", mode="lines", hovertemplate="%{y:.1f}%"))
    fig.update_layout(
        title="Kaggriculture top-band lineage occupancy (seats/day)<br>"
              "<sup>seats sampled: "
              + ", ".join(f"{d[5:]}:{ns[d]}" for d in dates) + "</sup>",
        yaxis=dict(range=[0, 100], title="share of seats", ticksuffix="%"),
        hovermode="x unified", height=640,
        # 凡例はプロット領域の外(下)。yanchor=top + 十分な下余白で重なり回避
        legend=dict(orientation="h", yanchor="top", y=-0.10, x=0,
                    font=dict(size=11)),
        margin=dict(t=80, b=30))
    fig.write_html(f"{OUT}/census_trend.html", include_plotlyjs="cdn")
    print(f"wrote {OUT}/census_trend.html", flush=True)
    fig.show()


def main():
    ds = []
    for pat in ("/kaggle/input/kaggriculture-episodes-*",
                "/kaggle/input/*/kaggriculture-episodes-*",
                "/kaggle/input/*/*/kaggriculture-episodes-*"):
        ds += [d for d in glob.glob(pat) if os.path.isdir(d)]
    history = {d: dict(c) for d, c in HISTORY.items()}
    for d in sorted(set(ds)):
        m = re.search(r"episodes-(\d{4}-\d{2}-\d{2})$", d)
        if not m:
            continue
        files = glob.glob(f"{d}/*.json") or glob.glob(f"{d}/*/*.json")
        print(f"{d}: {len(files)} files", flush=True)
        census = mine_date(m.group(1), files)
        counts = {}
        for row in census:
            for lin in row["lineages"]:
                counts[lin] = counts.get(lin, 0) + 1
        history[m.group(1)] = counts
    render_trend(history)


main()
